# Transformer from Scratch — "Attention Is All You Need"

This notebook implements the **Transformer** architecture from scratch in PyTorch. We build every component from the ground up and train it on English→Hindi translation (same task as `english_hindi_translation.ipynb`).

## Prerequisites
You should already understand:
- RNN / LSTM / GRU basics (`rnn_basics.ipynb`)
- Encoder-Decoder architecture (`english_hindi_translation.ipynb`)
- Bahdanau (additive) attention

## What's New in the Transformer?
| RNN Encoder-Decoder | Transformer |
|---|---|
| Sequential processing (slow) | Fully parallel (fast) |
| Attention + recurrence | Attention **only** |
| Vanishing gradient risk | No gradient issues |
| Hidden state carries info | Positional encoding + self-attention |

## Architecture Overview
```
Input → Embedding + Positional Encoding
         ↓
    ┌─────────────────┐
    │   ENCODER × N   │
    │ ┌─────────────┐ │
    │ │ Multi-Head   │ │  ← Self-Attention (Q=K=V from same source)
    │ │ Self-Attn    │ │
    │ └──────┬──────┘ │
    │    Add & Norm   │  ← Residual + LayerNorm
    │ ┌──────┴──────┐ │
    │ │ Feed-Forward │ │  ← FFN(x) = ReLU(xW₁+b₁)W₂+b₂
    │ └──────┬──────┘ │
    │    Add & Norm   │
    └─────────────────┘
         ↓
    ┌─────────────────────────────┐
    │       DECODER × N           │
    │ ┌─────────────────────────┐ │
    │ │ Masked Multi-Head       │ │  ← Can't peek at future tokens
    │ │ Self-Attention          │ │
    │ └───────────┬─────────────┘ │
    │       Add & Norm            │
    │ ┌───────────┴─────────────┐ │
    │ │ Cross-Attention         │ │  ← Q from decoder, K,V from encoder
    │ │ (Encoder-Decoder Attn)  │ │
    │ └───────────┬─────────────┘ │
    │       Add & Norm            │
    │ ┌───────────┴─────────────┐ │
    │ │ Feed-Forward            │ │
    │ └───────────┬─────────────┘ │
    │       Add & Norm            │
    └─────────────────────────────┘
         ↓
      Linear → Softmax → Output Token
```

## 1. Install & Import Dependencies

In [ ]:
!pip install torch datasets sacremoses matplotlib -q

In [ ]:
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load and Prepare Data

Same IITB English-Hindi dataset as `english_hindi_translation.ipynb`.

In [ ]:
# Load IIT Bombay English-Hindi dataset
dataset = load_dataset("cfilt/iitb-english-hindi")
print(dataset)

# Use subset for faster training
MAX_SAMPLES = 100000
train_data = dataset['train']['translation'][:MAX_SAMPLES]

english_sentences = [item['en'] for item in train_data]
hindi_sentences = [item['hi'] for item in train_data]

print(f"Total training samples: {len(english_sentences)}")
print(f"Sample English: {english_sentences[0]}")
print(f"Sample Hindi: {hindi_sentences[0]}")

## 3. Tokenizer (Character-level)

Same as in `english_hindi_translation.ipynb`. We use character-level tokenization for simplicity.

In [ ]:
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.PAD_token = 0
        self.SOS_token = 1
        self.EOS_token = 2
        self.UNK_token = 3

        self.token2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2token = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.n_tokens = 4

    def add_sentence(self, sentence):
        for char in sentence:
            if char not in self.token2idx:
                self.token2idx[char] = self.n_tokens
                self.idx2token[self.n_tokens] = char
                self.n_tokens += 1

    def encode(self, sentence, max_len=None):
        indices = [self.token2idx.get(c, self.UNK_token) for c in sentence]
        indices.append(self.EOS_token)
        if max_len:
            indices = indices[:max_len]
            indices += [self.PAD_token] * (max_len - len(indices))
        return indices

    def decode(self, indices):
        chars = []
        for idx in indices:
            if idx == self.EOS_token:
                break
            if idx not in [self.PAD_token, self.SOS_token]:
                chars.append(self.idx2token.get(idx, '<UNK>'))
        return ''.join(chars)

# Build vocabularies
eng_vocab = Vocabulary('english')
hin_vocab = Vocabulary('hindi')

print("Building vocabularies...")
for sent in english_sentences:
    eng_vocab.add_sentence(sent)
for sent in hindi_sentences:
    hin_vocab.add_sentence(sent)

print(f"English vocab size: {eng_vocab.n_tokens}")
print(f"Hindi vocab size: {hin_vocab.n_tokens}")

## 4. Dataset & DataLoader

In [ ]:
MAX_SEQ_LEN = 100

class TranslationDataset(Dataset):
    def __init__(self, english, hindi, eng_vocab, hin_vocab, max_len):
        self.english = english
        self.hindi = hindi
        self.eng_vocab = eng_vocab
        self.hin_vocab = hin_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.english)

    def __getitem__(self, idx):
        eng = self.english[idx]
        hin = self.hindi[idx]

        eng_enc = self.eng_vocab.encode(eng, self.max_len)
        hin_enc = self.hin_vocab.encode(hin, self.max_len)

        return (
            torch.tensor(eng_enc, dtype=torch.long),
            torch.tensor(hin_enc, dtype=torch.long)
        )

dataset_obj = TranslationDataset(english_sentences, hindi_sentences, eng_vocab, hin_vocab, MAX_SEQ_LEN)

train_size = int(0.95 * len(dataset_obj))
val_size = len(dataset_obj) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset_obj, [train_size, val_size])

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 5. Scaled Dot-Product Attention

This is the **core building block** of the Transformer. In your RNN notebook, you used **additive (Bahdanau) attention**:
```
energy = tanh(W1·decoder_hidden + W2·encoder_output)
score = v·energy
```

The Transformer uses **dot-product attention** instead — it's faster because it uses matrix multiplication:
```
Attention(Q, K, V) = softmax(QK^T / √d_k) · V
```

Why divide by √d_k? When d_k is large, dot products grow large in magnitude, pushing softmax into regions with tiny gradients. Scaling prevents this.

### Connection to your Bahdanau attention:
| Bahdanau (additive) | Scaled Dot-Product |
|---|---|
| `tanh(W·concat(h, enc))` | `QK^T / √d_k` |
| Learned scoring function | Dot product (similarity) |
| Slower (extra linear layers) | Faster (just matmul) |
| No scaling needed | Divide by √d_k |

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """
    Scaled Dot-Product Attention from the paper:
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        Q: Queries  (batch, seq_len_q, d_k)
        K: Keys     (batch, seq_len_k, d_k)
        V: Values   (batch, seq_len_k, d_v)
        mask: Optional mask to prevent attention to certain positions
    
    Returns:
        output: (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)  # dimension of keys
        
        # Step 1: Compute attention scores
        # Q: (batch, seq_len_q, d_k), K^T: (batch, d_k, seq_len_k)
        # scores: (batch, seq_len_q, seq_len_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Step 2: Apply mask (if provided)
        # Masked positions get -inf so softmax gives them ~0 attention
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Step 3: Softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Step 4: Weighted sum of values
        # attention_weights: (batch, seq_len_q, seq_len_k)
        # V: (batch, seq_len_k, d_v)
        # output: (batch, seq_len_q, d_v)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights


# Test it!
batch_size, seq_len, d_k = 2, 5, 64
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

attn = ScaledDotProductAttention()
output, weights = attn(Q, K, V)

print(f"Q shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (each row sums to 1):")
print(weights[0].detach().numpy().round(3))

## 6. Multi-Head Attention

Instead of one big attention with d_model=512, the paper splits into **h=8 parallel heads**, each with d_k = d_v = d_model/h = 64.

```
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) · W_O
where head_i = Attention(Q·W_i^Q, K·W_i^K, V·W_i^V)
```

**Why multiple heads?** Each head can learn different attention patterns:
- Head 1 might attend to the previous word (syntax)
- Head 2 might attend to related words (semantics)
- Head 3 might attend to punctuation

This is like having multiple "perspectives" on the same input.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from the paper.
    
    We use h=8 heads, each with d_k = d_v = d_model/h = 64.
    
    This can be used for:
    1. Encoder self-attention: Q=K=V from previous encoder layer
    2. Decoder masked self-attention: Q=K=V from previous decoder layer (with mask)
    3. Cross-attention: Q from decoder, K=V from encoder output
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head
        
        # Linear projections for Q, K, V and output
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(dropout)
        
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Step 1: Linear projections
        # (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        # Step 2: Split into h heads
        # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Step 3: Apply scaled dot-product attention for each head
        # (batch, num_heads, seq_len_q, d_k)
        attn_output, attn_weights = self.attention(Q, K, V, mask)
        
        # Step 4: Concatenate heads
        # (batch, num_heads, seq_len_q, d_k) -> (batch, seq_len_q, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        # Step 5: Final linear projection
        output = self.W_O(attn_output)
        
        return output, attn_weights


# Test Multi-Head Attention
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)

# Self-attention (Q=K=V=x)
output, weights = mha(x, x, x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}  (batch, heads, seq, seq)")
print(f"Number of parameters: {sum(p.numel() for p in mha.parameters()):,}")

## 7. Positional Encoding

**The problem:** RNNs naturally know word order because they process sequentially. Transformers process all words in parallel, so they have **no idea** about word order by default!

**The solution:** Add positional encodings to the input embeddings. The paper uses **sinusoidal functions**:
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Why sinusoidal?
1. Each dimension oscillates at a different frequency
2. The model can learn to attend by relative positions (PE(pos+k) can be represented as a linear function of PE(pos))
3. Can generalize to sequence lengths longer than training

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding from the paper.
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices
        
        pe = pe.unsqueeze(0)  # (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)  # not a trainable parameter
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Visualize positional encoding
pe = PositionalEncoding(d_model=128, max_seq_len=100)
x = torch.zeros(1, 100, 128)
pe_values = pe(x)[0].detach().numpy()

plt.figure(figsize=(12, 6))
plt.imshow(pe_values.T, aspect='auto', cmap='RdBu')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding (d_model=128)')
plt.colorbar()
plt.show()

print("Notice: Each dimension oscillates at a different frequency!")
print("Lower dimensions = high frequency (change fast)")
print("Higher dimensions = low frequency (change slowly)")

## 8. Position-wise Feed-Forward Network

Each encoder/decoder layer has a **feed-forward network** applied to each position independently:
```
FFN(x) = max(0, x·W₁ + b₁)·W₂ + b₂
```

This is just two linear layers with ReLU activation. It processes each position **separately** (same weights, applied to each time step).

| RNN | Transformer FFN |
|---|---|
| Recurrent layer processes sequence | Self-attention already did that |
| Hidden state carries info | FFN adds non-linearity to each position |
| Shares info across time | Independent per position |

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """
    FFN(x) = max(0, xW1 + b1)W2 + b2
    
    d_model: input/output dimension (512)
    d_ff: inner dimension (2048)
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


# Test
ffn = PositionwiseFeedForward(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)
output = ffn(x)
print(f"Input:  {x.shape}")
print(f"Output: {output.shape}")
print(f"Parameters: {sum(p.numel() for p in ffn.parameters()):,}")

## 9. Encoder Layer & Stack

Each encoder layer has:
1. **Multi-Head Self-Attention** (Q=K=V from previous layer)
2. **Add & LayerNorm** (residual connection + normalization)
3. **Feed-Forward Network**
4. **Add & LayerNorm**

The paper stacks **N=6** identical encoder layers.

In [ ]:
class EncoderLayer(nn.Module):
    """
    Single encoder layer:
    1. Multi-Head Self-Attention
    2. Add & LayerNorm
    3. Feed-Forward Network
    4. Add & LayerNorm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        # 1. Self-attention with residual + layer norm
        attn_output, _ = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # 2. Feed-forward with residual + layer norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        
        return x


class Encoder(nn.Module):
    """
    Encoder stack: N identical layers.
    
    Paper uses: N=6, d_model=512, h=8, d_ff=2048
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.d_model = d_model
    
    def forward(self, src, src_mask=None):
        # Embedding + positional encoding
        # Scale embeddings by sqrt(d_model) as per paper
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through N encoder layers
        for layer in self.layers:
            x = layer(x, src_mask)
        
        return x


# Test Encoder
encoder = Encoder(
    vocab_size=eng_vocab.n_tokens,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    max_seq_len=MAX_SEQ_LEN
)

src = torch.randint(0, eng_vocab.n_tokens, (2, 20))
enc_output = encoder(src)
print(f"Source shape: {src.shape}")
print(f"Encoder output shape: {enc_output.shape}")
print(f"Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")

## 10. Decoder Layer & Stack

Each decoder layer has **three** sub-layers (vs two in encoder):
1. **Masked Multi-Head Self-Attention** — can't peek at future tokens!
2. **Add & LayerNorm**
3. **Cross-Attention** — Q from decoder, K,V from encoder output
4. **Add & LayerNorm**
5. **Feed-Forward Network**
6. **Add & LayerNorm**

### The Causal Mask
In the decoder, position `i` can only attend to positions `≤ i`. This prevents "cheating" by looking at future tokens during training.
```
Mask for seq_len=4:
[1, 0, 0, 0]  ← position 0 can only see itself
[1, 1, 0, 0]  ← position 1 can see 0 and 1
[1, 1, 1, 0]  ← position 2 can see 0, 1, and 2
[1, 1, 1, 1]  ← position 3 can see everything
```

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal (look-ahead) mask for the decoder.
    Prevents position i from attending to positions > i.
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)


def create_padding_mask(seq, pad_idx=0):
    """
    Create a padding mask. Mask out PAD tokens.
    """
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)


# Visualize causal mask
causal_mask = create_causal_mask(8)[0, 0]
print("Causal mask (1 = can attend, 0 = masked):")
print(causal_mask.int().numpy())

plt.figure(figsize=(6, 6))
plt.imshow(causal_mask, cmap='Blues')
plt.title('Causal Mask (Lower Triangular)')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.show()

In [ ]:
class DecoderLayer(nn.Module):
    """
    Single decoder layer with THREE sub-layers:
    1. Masked Multi-Head Self-Attention
    2. Cross-Attention (encoder-decoder attention)
    3. Feed-Forward Network
    
    Each with residual connection + layer normalization.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # Sub-layer 1: Masked self-attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Sub-layer 2: Cross-attention (Q from decoder, K,V from encoder)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        
        # Sub-layer 3: Feed-forward
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention (decoder can't see future tokens)
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # 2. Cross-attention (decoder attends to encoder output)
        # Q comes from decoder, K and V come from encoder
        attn_output, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout2(attn_output))
        
        # 3. Feed-forward
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        
        return x


class Decoder(nn.Module):
    """
    Decoder stack: N identical layers.
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.d_model = d_model
    
    def forward(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        # Embedding + positional encoding
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through N decoder layers
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        return x


# Test Decoder
decoder = Decoder(
    vocab_size=hin_vocab.n_tokens,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    max_seq_len=MAX_SEQ_LEN
)

tgt = torch.randint(0, hin_vocab.n_tokens, (2, 15))
tgt_mask = create_causal_mask(15)
dec_output = decoder(tgt, enc_output, tgt_mask=tgt_mask)

print(f"Target shape: {tgt.shape}")
print(f"Encoder output shape: {enc_output.shape}")
print(f"Decoder output shape: {dec_output.shape}")
print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

## 11. Full Transformer Model

Now we put it all together! The complete Transformer:

1. **Encoder**: src tokens → embeddings + PE → N encoder layers → encoder output
2. **Decoder**: tgt tokens → embeddings + PE → N decoder layers (with cross-attention to encoder output) → decoder output
3. **Output projection**: decoder output → linear layer → vocabulary logits

### Paper's hyperparameters:
| Hyperparameter | Base Model | Big Model |
|---|---|---|
| d_model | 512 | 1024 |
| d_ff | 2048 | 4096 |
| h (heads) | 8 | 16 |
| N (layers) | 6 | 6 |
| dropout | 0.1 | 0.3 |

In [ ]:
class Transformer(nn.Module):
    """
    Full Transformer model for sequence-to-sequence translation.
    
    Architecture:
    - Encoder: processes source sequence
    - Decoder: generates target sequence (auto-regressively)
    - Output projection: decoder output → vocabulary logits
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, max_seq_len=100, dropout=0.1):
        super().__init__()
        
        self.encoder = Encoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len, dropout)
        
        # Final linear layer to project to vocabulary
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        # Initialize weights (important for training stability)
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # 1. Encode source sequence
        encoder_output = self.encoder(src, src_mask)
        
        # 2. Decode target sequence (attending to encoder output)
        decoder_output = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        
        # 3. Project to vocabulary
        output = self.fc_out(decoder_output)
        
        return output
    
    def translate(self, src, max_len=100, sos_idx=1, eos_idx=2):
        """
        Greedy decoding for inference.
        Generate output token by token, feeding each prediction back as input.
        """
        self.eval()
        with torch.no_grad():
            # Encode source once
            encoder_output = self.encoder(src)
            
            # Start with SOS token
            tgt = torch.tensor([[sos_idx]], dtype=torch.long, device=src.device)
            
            result = []
            attention_weights = []
            
            for _ in range(max_len):
                # Create causal mask for current sequence
                tgt_mask = create_causal_mask(tgt.size(1)).to(src.device)
                
                # Decode
                decoder_output = self.decoder(tgt, encoder_output, tgt_mask=tgt_mask)
                output = self.fc_out(decoder_output)
                
                # Get last token prediction
                next_token = output[:, -1, :].argmax(dim=-1).item()
                result.append(next_token)
                
                if next_token == eos_idx:
                    break
                
                # Append to tgt for next iteration
                tgt = torch.cat([tgt, torch.tensor([[next_token]], device=src.device)], dim=1)
            
            return result


# Create the full model
model = Transformer(
    src_vocab_size=eng_vocab.n_tokens,
    tgt_vocab_size=hin_vocab.n_tokens,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nModel architecture:")
print(model)

## 12. Training

We train with:
- **Adam optimizer** with warmup (as per paper)
- **Cross-entropy loss** (ignoring PAD tokens)
- **Teacher forcing**: feed ground truth as decoder input during training

The paper's learning rate schedule:
```
lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
```
This increases linearly for warmup_steps=4000, then decreases.

In [ ]:
class WarmupLRScheduler:
    """
    Learning rate scheduler from the paper:
    lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
    """
    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0
    
    def step(self):
        self.step_num += 1
        lr = self.d_model ** (-0.5) * min(
            self.step_num ** (-0.5),
            self.step_num * self.warmup_steps ** (-1.5)
        )
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr


# Loss function (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Optimizer with paper's settings
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = WarmupLRScheduler(optimizer, d_model=512, warmup_steps=4000)

print("Optimizer and scheduler ready!")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    
    for src, tgt in tqdm(loader, desc="Training"):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Create causal mask for target
        tgt_mask = create_causal_mask(tgt.size(1)).to(device)
        
        # Teacher forcing: input is tgt[:-1], target is tgt[1:]
        tgt_input = tgt[:, :-1]   # everything except last token
        tgt_output = tgt[:, 1:]   # everything except first token (SOS)
        
        tgt_mask = create_causal_mask(tgt_input.size(1)).to(device)
        
        optimizer.zero_grad()
        
        output = model(src, tgt_input, tgt_mask=tgt_mask)
        
        # Reshape for loss
        output = output.reshape(-1, output.shape[-1])
        tgt_output = tgt_output.reshape(-1)
        
        loss = criterion(output, tgt_output)
        loss.backward()
        
        # Gradient clipping (paper uses 1.0)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, tgt in tqdm(loader, desc="Evaluating"):
            src = src.to(device)
            tgt = tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            tgt_mask = create_causal_mask(tgt_input.size(1)).to(device)
            
            output = model(src, tgt_input, tgt_mask=tgt_mask)
            
            output = output.reshape(-1, output.shape[-1])
            tgt_output = tgt_output.reshape(-1)
            
            loss = criterion(output, tgt_output)
            total_loss += loss.item()
    
    return total_loss / len(loader)


# Training loop
NUM_EPOCHS = 10
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{NUM_EPOCHS} ===")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'transformer_best.pt')
        print("Saved best model!")

## 13. Inference & Translation

Let's test our trained Transformer on some English→Hindi translations!

In [ ]:
# Load best model
model.load_state_dict(torch.load('transformer_best.pt'))

def translate_sentence(sentence, model, eng_vocab, hin_vocab, max_len=100):
    model.eval()
    tokens = eng_vocab.encode(sentence, max_len)
    src = torch.tensor(tokens).unsqueeze(0).to(device)
    
    result = model.translate(src, max_len=max_len)
    hindi_translation = hin_vocab.decode(result)
    
    return hindi_translation


# Test translations
test_samples = [
    "hello how are you",
    "i love machine learning",
    "what is your name",
    "good morning",
    "thank you very much"
]

print("=== Transformer Translations ===\n")
for sample in test_samples:
    translation = translate_sentence(sample, model, eng_vocab, hin_vocab)
    print(f"English: {sample}")
    print(f"Hindi:   {translation}\n")

## 14. Attention Visualization

One of the coolest things about Transformers: we can visualize what each attention head is looking at!

We'll visualize:
1. **Encoder self-attention**: how source words attend to each other
2. **Decoder self-attention**: how generated words attend to previous words
3. **Cross-attention**: how decoder attends to source (this is like your Bahdanau attention visualization!)

In [ ]:
def get_attention_weights(model, src, tgt):
    """
    Extract attention weights from all layers/heads for visualization.
    """
    model.eval()
    
    # We need to modify the forward pass to capture attention weights
    # For simplicity, let's visualize using the model's internal structure
    
    src_mask = None
    tgt_mask = create_causal_mask(tgt.size(1)).to(src.device)
    
    # Encode
    enc_output = model.encoder(src, src_mask)
    
    # Collect attention weights from first decoder layer
    x = model.decoder.embedding(tgt) * math.sqrt(model.decoder.d_model)
    x = model.decoder.pos_encoding(x)
    
    # Get attention from first decoder layer
    layer = model.decoder.layers[0]
    
    # Self-attention
    self_attn_output, self_attn_weights = layer.self_attn(x, x, x, tgt_mask)
    
    # Cross-attention
    normed = layer.norm1(x + layer.dropout1(self_attn_output))
    cross_attn_output, cross_attn_weights = layer.cross_attn(normed, enc_output, enc_output, src_mask)
    
    return self_attn_weights, cross_attn_weights


# Visualize attention for a sample sentence
sample = "hello how are you"
translation = translate_sentence(sample, model, eng_vocab, hin_vocab)

# Encode source and target
src_tokens = eng_vocab.encode(sample, MAX_SEQ_LEN)
tgt_tokens = [hin_vocab.SOS_token] + hin_vocab.encode(translation, MAX_SEQ_LEN-1)

src_tensor = torch.tensor(src_tokens).unsqueeze(0).to(device)
tgt_tensor = torch.tensor(tgt_tokens).unsqueeze(0).to(device)

self_attn, cross_attn = get_attention_weights(model, src_tensor, tgt_tensor)

# Plot cross-attention (most interesting - shows how decoder attends to source)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Cross-Attention Heads (Decoder → Encoder)', fontsize=14)

src_chars = list(sample)[:20]
tgt_chars = list(translation)[:15]

for head_idx in range(8):
    ax = axes[head_idx // 4, head_idx % 4]
    attn = cross_attn[0, head_idx, :len(tgt_chars), :len(src_chars)].cpu().detach().numpy()
    ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(src_chars)))
    ax.set_yticks(range(len(tgt_chars)))
    ax.set_xticklabels(src_chars, fontsize=7, rotation=45)
    ax.set_yticklabels(tgt_chars, fontsize=7)
    ax.set_title(f'Head {head_idx+1}')

plt.tight_layout()
plt.show()

print(f"Source: {sample}")
print(f"Translation: {translation}")

## 15. Summary: RNN+Attention vs Transformer

| Aspect | RNN + Bahdanau | Transformer |
|---|---|---|
| **Processing** | Sequential (word by word) | Parallel (all words at once) |
| **Attention type** | Additive (learned scoring) | Dot-product (similarity) |
| **Position info** | Implicit (recurrence) | Explicit (positional encoding) |
| **Long-range deps** | Struggles (vanishing gradient) | O(1) path length |
| **Training speed** | Slow (can't parallelize) | Fast (fully parallelizable) |
| **Interpretability** | Single attention | Multiple attention heads |
| **Model size** | Usually smaller | Usually larger |

### Key Takeaways
1. **Attention replaces recurrence** — no more sequential bottleneck
2. **Multi-head attention** = multiple "perspectives" in parallel
3. **Positional encoding** = inject order information (since no recurrence)
4. **Residual connections + LayerNorm** = stable training for deep networks
5. **Causal mask** = prevents decoder from "cheating" by seeing future tokens

### What's next?
- **BERT** (2018): Encoder-only Transformer for understanding
- **GPT** (2018-): Decoder-only Transformer for generation
- **T5** (2019): Encoder-decoder Transformer (text-to-text)
- **Vision Transformer (ViT)**: Transformer for images!